### Computer Vision: Convolutional Neural Networks (CNN) untuk Klasifikasi Gambar

Sekarang, mari kita buat model canggih menggunakan Convolutional Neural Networks (CNN) untuk klasifikasi gambar. Kita akan menggunakan dataset Fashion MNIST, yang berisi gambar-gambar artikel pakaian.

Pertama, kita akan mengimpor library yang diperlukan, termasuk TensorFlow dan Keras.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

### Memuat dan Mempersiapkan Dataset Fashion MNIST

Dataset Fashion MNIST terdiri dari 60.000 gambar grayscale berukuran 28x28 piksel untuk pelatihan dan 10.000 gambar untuk pengujian, yang dikategorikan ke dalam 10 kelas.

In [ ]:
# Muat dataset Fashion MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalisasi gambar ke skala 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape data untuk input CNN (menambah dimensi channel tunggal)
x_train = np.expand_dims(x_train, -1) # Menjadi (60000, 28, 28, 1)
x_test = np.expand_dims(x_test, -1)   # Menjadi (10000, 28, 28, 1)

print(f"Bentuk data pelatihan: {x_train.shape}")
print(f"Bentuk label pelatihan: {y_train.shape}")
print(f"Bentuk data pengujian: {x_test.shape}")
print(f"Bentuk label pengujian: {y_test.shape}")

# Definisikan nama-nama kelas
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Tampilkan beberapa contoh gambar
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[y_train[i]])
plt.show()

### Membangun Model CNN

Kita akan membangun model CNN sederhana dengan beberapa lapisan konvolusi, lapisan pooling, dan lapisan dense (fully connected) untuk klasifikasi.

In [ ]:
model = keras.Sequential(
    [
        keras.Input(shape=(28, 28, 1)), # Input layer sesuai dimensi gambar
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(), # Meratakan output dari lapisan konvolusi
        layers.Dropout(0.5), # Regularisasi untuk mencegah overfitting
        layers.Dense(10, activation="softmax"), # Output layer dengan 10 kelas
    ]
)

model.summary()

### Melatih Model CNN

Selanjutnya, kita akan mengompilasi dan melatih model menggunakan data pelatihan.

In [ ]:
batch_size = 128
epochs = 5 # Anda bisa menambah ini untuk akurasi yang lebih baik

model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

# Plotting training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

### Mengevaluasi Model

Kita akan mengevaluasi kinerja model menggunakan data pengujian.

In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {score[0]:.4f}")
print(f"Test accuracy: {score[1]:.4f}")

### Melakukan Prediksi

Mari kita coba memprediksi beberapa gambar dari dataset pengujian.

In [ ]:
predictions = model.predict(x_test)

def plot_image(i, predictions_array, true_label, img):
    true_label, img = true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])

    plt.imshow(img.reshape(28, 28), cmap=plt.cm.binary)

    predicted_label = np.argmax(predictions_array)
    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'

    plt.xlabel(f"{class_names[predicted_label]} {100*np.max(predictions_array):2.0f}% (True: {class_names[true_label]})",
               color=color)

def plot_value_array(i, predictions_array, true_label):
    true_label = true_label[i]
    plt.grid(False)
    plt.xticks(range(10), class_names, rotation=90)
    plt.yticks([])
    thisplot = plt.bar(range(10), predictions_array, color="#777777")
    plt.ylim([0, 1])
    predicted_label = np.argmax(predictions_array)

    thisplot[predicted_label].set_color('red')
    thisplot[true_label].set_color('blue')

num_rows = 5
num_cols = 3
num_images = num_rows * num_cols
plt.figure(figsize=(2 * 2 * num_cols, 2 * num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 1)
    plot_image(i, predictions[i], y_test, x_test)
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 2)
    plot_value_array(i, predictions[i], y_test)
plt.tight_layout()
plt.show()

### Uji Model dengan Gambar yang Diunggah

Anda dapat mengunggah gambar Anda sendiri untuk melihat bagaimana model CNN ini memprediksinya.

#### Langkah 1: Unggah Gambar

Jalankan sel di bawah ini, lalu klik 'Choose Files' untuk memilih gambar dari komputer Anda. Pastikan gambar yang Anda unggah adalah gambar grayscale atau akan dikonversi menjadi grayscale.

In [ ]:
from google.colab import files
import io
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt

uploaded = files.upload()

if uploaded:
    # Ambil nama file yang diunggah (hanya proses satu file)
    uploaded_image_name = list(uploaded.keys())[0]
    print(f'User uploaded file "{uploaded_image_name}"')

    # Baca gambar yang diunggah
    img = Image.open(io.BytesIO(uploaded[uploaded_image_name]))

    # Konversi ke grayscale jika belum (Fashion MNIST adalah grayscale)
    if img.mode != 'L':
        img = img.convert('L')

    # Resize gambar ke 28x28 piksel
    img = img.resize((28, 28))

    # Konversi ke array numpy
    img_array = np.array(img, dtype="float32")

    # Normalisasi (skala 0-1)
    img_array = img_array / 255.0

    # Reshape untuk input model CNN (menambah dimensi batch dan channel)
    # Dari (28, 28) menjadi (1, 28, 28, 1)
    processed_image = np.expand_dims(img_array, axis=0)
    processed_image = np.expand_dims(processed_image, axis=-1)

    # Lakukan prediksi
    uploaded_prediction = model.predict(processed_image)
    predicted_class = np.argmax(uploaded_prediction[0])
    predicted_confidence = np.max(uploaded_prediction[0])

    # Tampilkan gambar dan prediksinya
    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img_array, cmap=plt.cm.binary)
    plt.xticks([])
    plt.yticks([])
    plt.title(f"Unggah: {uploaded_image_name}")

    plt.subplot(1, 2, 2)
    plt.bar(range(10), uploaded_prediction[0], color="#777777")
    plt.ylim([0, 1])
    plt.xticks(range(10), class_names, rotation=90)
    plt.title("Probabilitas Prediksi")
    plt.text(predicted_class, predicted_confidence + 0.1,
             f'{class_names[predicted_class]} ({predicted_confidence*100:.0f}%)',
             ha='center', va='bottom', color='red', fontsize=10)

    plt.tight_layout()
    plt.show()

    print(f"Gambar terunggah diprediksi sebagai: {class_names[predicted_class]} dengan keyakinan {predicted_confidence*100:.2f}%")
else:
    print("Tidak ada gambar yang diunggah.")